# Phase 10 — Production Deployment and Operations

- **10A.** Repository and command inspection
- **10B.** Cloud decision and resource inventory
- **10C.** Production configuration
- **10D.** Object-storage artifact repository
- **10E.** Structured logging and reports
- **10F.** CI
- **10G.** Production container images
- **10H.** Registry publishing
- **10I.** Staging infrastructure and deployment
- **10J.** Scheduled inference/AQI pipeline
- **10K.** Incremental and retraining schedules
- **10L.** Monitoring, alerts, and stale-data checks
- **10M.** Production deployment
- **10N.** Rollback and failure testing
- **10O.** Documentation and final reports

## **10A.** Repository and operational-command inspection

Phase 10A validates that the existing application and MLOps workloads can be
operated non-interactively before cloud infrastructure or GitHub Actions are
created.

The inspection covers:

- serving applications
- Docker assets
- dependency management
- automated tests
- batch-pipeline commands
- Hopsworks commands
- artifact directories
- structured reports
- health endpoints
- existing workflow files

No deployment or cloud resource is created during this subphase.

In [2]:
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (
    PROJECT_ROOT / "pyproject.toml"
).exists(), "Could not resolve the project root."

print("Project root:", PROJECT_ROOT)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor


### Deployment asset inspection

The repository is inspected for the minimum assets required before selecting a
cloud deployment target:

- FastAPI Dockerfile
- Streamlit Dockerfile
- Docker Compose configuration
- locked Python dependencies
- environment template
- tests
- serving entry points
- reusable Phase 9 batch commands

In [2]:
from app.operations.repository_inspection import (
    inspect_expected_files,
)

import pandas as pd


file_inspections = inspect_expected_files()

file_inspection_df = pd.DataFrame(
    [
        {
            "name": item.name,
            "path": item.path,
            "exists": item.exists,
            "required": item.required,
        }
        for item in file_inspections
    ]
)

display(file_inspection_df)

,name,path,exists,required
0,FastAPI Dockerfile,Dockerfile,True,True
1,Streamlit Dockerfile,dashboard/Dockerfile,True,True
2,Docker Compose,compose.yaml,True,True
3,Python project configuration,pyproject.toml,True,True
4,Locked dependencies,uv.lock,True,True
5,Environment template,.env.example,True,True
6,FastAPI application,app/api/main.py,True,True
7,Streamlit application,dashboard/app.py,True,True
8,Live inference notebook,notebooks/06_live_inference_pipeline.ipynb,True,True
9,AQI pipeline notebook,notebooks/07_build_aqi_alert_pipeline.ipynb,False,True


### Operational command contract

Every production command must:

- run without interactive input
- produce a meaningful exit code
- use UTC timestamps
- write a structured report when it is a batch pipeline
- fail clearly
- avoid logging secrets
- remain safe to retry where applicable

Serving commands are long-running processes. Batch commands must terminate
after success or failure.

In [3]:
from app.operations.repository_inspection import (
    inspect_commands,
)


command_inspections = inspect_commands()

commands_df = pd.DataFrame(
    [
        {
            "name": item.name,
            "category": item.category,
            "available": item.available,
            "non_interactive": item.non_interactive,
            "command": item.command,
            "expected_report": item.expected_report,
        }
        for item in command_inspections
    ]
)

display(commands_df)

,name,category,available,non_interactive,command,expected_report
0,Run complete test suite,validation,True,True,uv run pytest -v,None
1,Run Ruff lint checks,validation,True,True,uv run ruff check .,None
2,Run Ruff formatting check,validation,True,True,uv run ruff format --check .,None
3,Start FastAPI,serving,True,True,uv run uvicorn app.api.main:app --host 0.0.0.0...,None
4,Start Streamlit,serving,True,True,uv run streamlit run dashboard/app.py --server...,None
5,Validate registry model resolution,mlops,True,True,uv run python -m app.pipelines.validate_regist...,reports/phase_9/production_model_resolution_re...
6,Run incremental feature synchronization,batch,True,True,uv run python -m app.pipelines.incremental_fea...,reports/phase_9/incremental_feature_report.json
7,Run historical backfill,batch,True,True,uv run python -m app.pipelines.historical_back...,reports/phase_9/historical_backfill_report.json
8,Build training dataset,mlops,True,True,uv run python -m app.pipelines.build_training_...,reports/phase_9/training_dataset_report.json
9,Run retraining eligibility,mlops,True,True,uv run python -m app.pipelines.retraining_cycle,reports/phase_9/automated_training_report.json


### Current artifact-storage boundary

The existing project stores validated artifacts under local directories such
as:

- `inference/runs`
- `inference/latest`
- `aqi/runs`
- `aqi/latest`
- `reports/phase_9`
- `models`

Local storage is suitable for development and Docker Compose, but independent
cloud services and scheduled jobs require durable shared storage.

Phase 10A records the current paths only. Object-storage implementation belongs
to a later Phase 10 subphase.

In [4]:
from app.operations.repository_inspection import (
    discover_artifact_directories,
)


artifact_summary = (
    discover_artifact_directories()
)

artifact_df = pd.DataFrame(
    [
        {
            "name": name,
            **details,
        }
        for name, details
        in artifact_summary.items()
    ]
)

display(artifact_df)

,name,path,exists,file_count
0,canonical_data,data/processed,True,3
1,training_data,data/training,True,8
2,models,models,True,30
3,inference_runs,inference/runs,True,48
4,inference_latest,inference/latest,False,0
5,aqi_runs,aqi/runs,True,30
6,aqi_latest,aqi/latest,True,6
7,phase_9_reports,reports/phase_9,True,10
8,phase_10_reports,reports/phase_10,True,3


### Existing health and serving endpoints

Deployment smoke tests will reuse the existing API and Streamlit health
contracts.

FastAPI must remain live even when forecast artifacts are missing or stale.
Readiness should represent artifact availability and freshness accurately.

In [5]:
from app.operations.repository_inspection import (
    HEALTH_ENDPOINTS,
)


health_endpoints_df = pd.DataFrame(
    HEALTH_ENDPOINTS
)

display(health_endpoints_df)

,name,path,expected_behavior
0,FastAPI liveness,/api/v1/health/live,HTTP 200
1,FastAPI readiness,/api/v1/health/ready,HTTP 200 when ready; structured non-ready resp...
2,Forecast,/api/v1/forecast,HTTP 200 with 72 rows when current artifacts a...
3,Forecast summary,/api/v1/forecast/summary,HTTP 200
4,Alerts,/api/v1/alerts,HTTP 200
5,Metadata,/api/v1/metadata,HTTP 200
6,Streamlit process health,/_stcore/health,HTTP 200


In [6]:
from app.operations.repository_inspection import (
    build_repository_operations_report,
    save_report,
)


repository_operations_report = (
    build_repository_operations_report()
)

repository_operations_report[
    "status"
]

'REPOSITORY_OPERATIONS_INSPECTION_INCOMPLETE'

### Manual Phase 10A review

The automated inspection confirms file and command availability, but the
following items require manual review:

1. whether live inference has a reusable non-notebook command
2. whether AQI and alert generation has a reusable non-notebook command
3. whether Phase 5 and Phase 6 publish artifacts atomically
4. whether all batch reports include a pipeline run ID
5. whether every failed command returns a non-zero exit code
6. whether routine tests avoid real external services
7. whether Docker images include unnecessary notebooks or datasets
8. whether production artifact paths can be redirected through configuration
9. whether any current GitHub workflow duplicates a future cloud scheduler
10. which cloud account, credits, and region are available

In [7]:
live_inference_runner = (
    PROJECT_ROOT
    / "app"
    / "pipelines"
    / "live_inference.py"
)

aqi_pipeline_runner = (
    PROJECT_ROOT
    / "app"
    / "pipelines"
    / "aqi_alert_pipeline.py"
)

identified_gaps = []

if not live_inference_runner.exists():
    identified_gaps.append(
        {
            "code": (
                "LIVE_INFERENCE_COMMAND_MISSING"
            ),
            "description": (
                "The reusable live inference runner "
                "does not exist."
            ),
        }
    )

if not aqi_pipeline_runner.exists():
    identified_gaps.append(
        {
            "code": (
                "AQI_PIPELINE_COMMAND_MISSING"
            ),
            "description": (
                "The reusable AQI and alert runner "
                "does not exist."
            ),
        }
    )

repository_operations_report[
    "identified_gaps"
] = identified_gaps

print(
    "Live inference runner:",
    live_inference_runner,
)

print(
    "AQI pipeline runner:",
    aqi_pipeline_runner,
)

print(
    "Identified gaps:",
    identified_gaps,
)

Live inference runner: /home/riyan/Riyan/projects/pearls-aqi-predictor/app/pipelines/live_inference.py
AQI pipeline runner: /home/riyan/Riyan/projects/pearls-aqi-predictor/app/pipelines/aqi_alert_pipeline.py
Identified gaps: []


In [8]:
REPORT_PATH = save_report(
    repository_operations_report
)

print(
    "Phase 10A report saved:",
    REPORT_PATH,
)

Phase 10A report saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/phase_10/repository_operations_report.json


## **10B.** Azure architecture

The project uses an Azure for Students subscription with limited credits.

To keep the deployment affordable, the first cloud version uses one shared
demo environment rather than separate staging and production environments.

The selected services are:

- Azure Container Apps for FastAPI and Streamlit
- Azure Container Apps Jobs for batch pipelines
- Azure Container Registry Basic
- Azure Blob Storage
- Azure Key Vault
- Azure Log Analytics

Kubernetes, a relational database, premium networking, and duplicate
environments are intentionally excluded.

### Minimum deployment architecture

The demo environment contains:

1. one resource group
2. one container registry
3. one Container Apps environment
4. one FastAPI container
5. one Streamlit container
6. one shared pipeline job
7. one storage account
8. one Key Vault
9. one Log Analytics workspace

The architecture can later be expanded into isolated staging and production
environments if additional budget becomes available.

### Cost-control decisions

The deployment uses:

- consumption-based Container Apps
- scale-to-zero where practical
- Basic Container Registry
- one shared pipeline image
- manual retraining and backfill
- limited log retention
- Blob Storage lifecycle policies
- no Kubernetes
- no database
- no automatic model promotion

The goal is to demonstrate production architecture without consuming the
student subscription unnecessarily.

In [9]:
import json
import subprocess


def run_azure_cli(
    arguments: list[str],
) -> object:
    result = subprocess.run(
        ["az", *arguments],
        check=True,
        capture_output=True,
        text=True,
    )

    return json.loads(result.stdout)


account = run_azure_cli(
    [
        "account",
        "show",
        "--output",
        "json",
    ]
)

{
    "subscription_name": account["name"],
    "subscription_state": account["state"],
    "tenant_id_available": bool(
        account.get("tenantId")
    ),
}

{'subscription_name': 'Azure for Students',
 'subscription_state': 'Enabled',
 'tenant_id_available': True}

## **10C.** Production configuration

Phase 10C defines environment-specific configuration for FastAPI, Streamlit,
and batch pipeline jobs.

Configuration is separated by service responsibility:

- FastAPI reads validated forecast artifacts
- Streamlit calls FastAPI only
- pipeline jobs access OpenAQ, Hopsworks, and artifact storage

Secrets are not committed to Git or embedded inside container images.

The Azure demo environment uses:

- Central India
- `rg-pearls-aqi-demo`
- the existing `walpole.azurecr.io` registry
- one future Blob Storage account
- one future Key Vault

### Configuration safety rules

The deployment configuration enforces:

- explicit service roles
- supported environments
- explicit CORS origins
- no wildcard CORS in cloud environments
- required storage settings for Azure Blob
- required OpenAQ credentials for pipeline jobs
- required Hopsworks settings when registry or feature-store mode is enabled
- automatic model promotion disabled
- no secret values written to reports

In [10]:
import importlib

import app.operations.deployment_config

importlib.reload(
    app.operations.deployment_config
)

from app.operations.deployment_config import (
    build_configuration_report,
)

In [11]:
api_test_environment = {
    "APP_ENV": "demo",
    "APP_VERSION": "test-sha",
    "SERVICE_ROLE": "api",
    "LOG_LEVEL": "INFO",
    "AZURE_LOCATION": "centralindia",
    "AZURE_RESOURCE_GROUP": "rg-pearls-aqi-demo",
    "AZURE_CONTAINER_REGISTRY": "walpole.azurecr.io",
    "ARTIFACT_BACKEND": "azure_blob",
    "AZURE_STORAGE_ACCOUNT": "stpearlsaqiriyan",
    "AZURE_STORAGE_CONTAINER": "artifacts",
    "AZURE_KEY_VAULT_NAME": "kv-pearls-aqi-riyan",
    "ALLOWED_CORS_ORIGINS": (
        "https://dashboard.example.com"
    ),
    "MODEL_LOADING_MODE": "LOCAL_ARTIFACT",
    "FEATURE_STORE_BACKEND": "local",
    "MODEL_REGISTRY_BACKEND": "local",
    "AUTOMATIC_MODEL_PROMOTION_ENABLED": "false",
}

api_configuration_report = (
    build_configuration_report(
        api_test_environment
    )
)

api_configuration_report["status"]

'DEPLOYMENT_CONFIGURATION_VALIDATED'

In [12]:
dashboard_test_environment = {
    "APP_ENV": "demo",
    "APP_VERSION": "test-sha",
    "SERVICE_ROLE": "dashboard",
    "LOG_LEVEL": "INFO",
    "FASTAPI_BASE_URL": "https://api.example.com",
    "ARTIFACT_BACKEND": "local",
    "MODEL_LOADING_MODE": "LOCAL_ARTIFACT",
    "FEATURE_STORE_BACKEND": "local",
    "MODEL_REGISTRY_BACKEND": "local",
    "AUTOMATIC_MODEL_PROMOTION_ENABLED": "false",
}

dashboard_configuration_report = (
    build_configuration_report(
        dashboard_test_environment
    )
)


pipeline_test_environment = {
    "APP_ENV": "demo",
    "APP_VERSION": "test-sha",
    "SERVICE_ROLE": "pipeline",
    "LOG_LEVEL": "INFO",
    "AZURE_LOCATION": "centralindia",
    "AZURE_RESOURCE_GROUP": "rg-pearls-aqi-demo",
    "AZURE_CONTAINER_REGISTRY": "walpole.azurecr.io",
    "ARTIFACT_BACKEND": "azure_blob",
    "AZURE_STORAGE_ACCOUNT": "stpearlsaqiriyan",
    "AZURE_STORAGE_CONTAINER": "artifacts",
    "AZURE_KEY_VAULT_NAME": "kv-pearls-aqi-riyan",
    "MODEL_LOADING_MODE": "HOPSWORKS_REGISTRY",
    "FEATURE_STORE_BACKEND": "hopsworks",
    "MODEL_REGISTRY_BACKEND": "hopsworks",
    "OPENAQ_API_KEY": "configured-test-value",
    "HOPSWORKS_API_KEY": "configured-test-value",
    "HOPSWORKS_PROJECT": "test-project",
    "HOPSWORKS_HOST": "test-host",
    "AUTOMATIC_RETRAINING_ENABLED": "false",
    "AUTOMATIC_MODEL_PROMOTION_ENABLED": "false",
}

pipeline_configuration_report = (
    build_configuration_report(
        pipeline_test_environment
    )
)


assert api_configuration_report[
    "approved"
]

assert dashboard_configuration_report[
    "approved"
]

assert pipeline_configuration_report[
    "approved"
]

assert not pipeline_configuration_report[
    "configuration"
][
    "automatic_model_promotion_enabled"
]

assert (
    pipeline_configuration_report[
        "secret_values_included"
    ]
    is False
)

print(
    "Phase 10C deployment configuration "
    "validation passed."
)

Phase 10C deployment configuration validation passed.


## **10D.** Durable artifact repository

Phase 10D introduces a shared artifact-storage contract for local development
and Azure deployment.

Local execution uses the filesystem.

Cloud execution will use Azure Blob Storage through passwordless authentication.

The repository stores immutable run directories and updates a small latest
pointer only after a complete run passes validation.

### Atomic publication boundary

Azure Blob Storage does not provide a transaction covering an entire directory.

The project therefore uses a pointer-based publication model:

1. upload every run artifact into a unique immutable prefix
2. calculate and record SHA-256 checksums
3. write the run manifest
4. update `latest/pointer.json` as the final operation

Readers discover only the run referenced by the latest pointer.

If a pipeline fails before the final pointer update, the previous successful
run remains active.

### Authentication model

The repository uses `DefaultAzureCredential`.

During local development it can use the authenticated Azure CLI session.

Inside Azure Container Apps it will use managed identity.

No Storage Account key or connection string is required in committed
configuration.

In [13]:
import importlib

import app.artifacts.repository
import app.operations.artifact_repository_validation

importlib.reload(
    app.artifacts.repository
)

importlib.reload(
    app.operations.artifact_repository_validation
)

from app.operations.artifact_repository_validation import (
    run_local_validation,
    save_validation_report,
)

In [14]:
artifact_repository_report = (
    run_local_validation()
)

artifact_repository_report

{'phase': '10D',
 'generated_at_utc': '2026-08-01T07:42:16.727666+00:00',
 'status': 'ARTIFACT_REPOSITORY_VALIDATED',
 'approved': True,
 'backend_validated': 'local',
 'azure_backend_implemented': True,
 'azure_live_connection_tested': False,
 'checks': {'run_manifest_exists': True,
  'latest_pointer_exists': True,
  'latest_points_to_run': True,
  'manifest_file_count': True,
  'all_manifest_checksums_present': True,
  'duplicate_run_blocked': True,
  'invalid_run_blocked': True},
 'publication_contract': {'immutable_run_paths': True,
  'checksums_recorded': True,
  'manifest_written_before_latest': True,
  'latest_written_last': True,
  'invalid_runs_cannot_be_latest': True},
 'azure_resources_created': False}

In [15]:
assert artifact_repository_report[
    "approved"
]

assert artifact_repository_report[
    "checks"
]["run_manifest_exists"]

assert artifact_repository_report[
    "checks"
]["latest_pointer_exists"]

assert artifact_repository_report[
    "checks"
]["duplicate_run_blocked"]

assert artifact_repository_report[
    "checks"
]["invalid_run_blocked"]

assert artifact_repository_report[
    "publication_contract"
]["latest_written_last"]

assert not artifact_repository_report[
    "azure_resources_created"
]

print(
    "Phase 10D artifact repository "
    "validation passed."
)

Phase 10D artifact repository validation passed.


In [16]:
ARTIFACT_REPORT_PATH = (
    save_validation_report(
        artifact_repository_report
    )
)

print(
    "Artifact repository report saved:",
    ARTIFACT_REPORT_PATH,
)

Artifact repository report saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/phase_10/artifact_repository_validation_report.json


## **10E.** Structured logging and operational reports

Phase 10E introduces one JSON logging contract for FastAPI and batch pipelines.

Structured records make it possible to search executions by:

- service
- pipeline
- pipeline run ID
- model version
- status
- error code
- duration
- row count

Logging uses standard output so Docker and Azure Container Apps can capture the
records without application-specific log files.

### Secret-safe logging

Secrets are never intentionally included in logs or reports.

Known credential-like fields are replaced with `[REDACTED]`.

The application must not log:

- complete environment dictionaries
- API keys
- authorization headers
- passwords
- access tokens
- Key Vault values
- Storage Account keys

In [17]:
import importlib

import app.observability.logging
import app.observability.reports
import app.operations.structured_logging_validation

importlib.reload(
    app.observability.logging
)

importlib.reload(
    app.observability.reports
)

importlib.reload(
    app.operations.structured_logging_validation
)

from app.operations.structured_logging_validation import (
    run_structured_logging_validation,
    save_validation_report,
)

In [18]:
structured_logging_report = (
    run_structured_logging_validation()
)

structured_logging_report

{'phase': '10E',
 'generated_at_utc': '2026-08-01T10:18:39.350208+00:00',
 'status': 'STRUCTURED_LOGGING_VALIDATED',
 'approved': True,
 'checks': {'log_is_valid_json': True,
  'timestamp_present': True,
  'level_present': True,
  'event_present': True,
  'run_id_present': True,
  'row_count_present': True,
  'log_secret_redacted': True,
  'report_secret_redacted': True,
  'nested_secret_redacted': True,
  'safe_value_preserved': True},
 'required_fields': ['timestamp_utc',
  'level',
  'service_name',
  'environment',
  'event',
  'pipeline_name',
  'pipeline_run_id',
  'status',
  'duration_seconds',
  'row_count',
  'model_version',
  'error_code'],
 'secret_values_included': False,
 'azure_resources_created': False}

In [19]:
assert structured_logging_report[
    "approved"
]

assert structured_logging_report[
    "checks"
]["log_is_valid_json"]

assert structured_logging_report[
    "checks"
]["log_secret_redacted"]

assert structured_logging_report[
    "checks"
]["report_secret_redacted"]

assert structured_logging_report[
    "checks"
]["nested_secret_redacted"]

assert not structured_logging_report[
    "secret_values_included"
]

print(
    "Phase 10E structured logging "
    "validation passed."
)

Phase 10E structured logging validation passed.


In [20]:
STRUCTURED_LOGGING_REPORT_PATH = (
    save_validation_report(
        structured_logging_report
    )
)

print(
    "Structured logging report saved:",
    STRUCTURED_LOGGING_REPORT_PATH,
)

Structured logging report saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/phase_10/structured_logging_validation_report.json


## **10F.** Continuous integration

Phase 10F introduces GitHub Actions validation for every pull request and push
to the main branch.

The workflow validates:

- the uv lockfile
- locked dependency installation
- targeted Ruff rules
- Python imports
- operational validation scripts
- automated tests
- Docker Compose configuration
- FastAPI image construction
- Streamlit image construction

Routine CI does not call external APIs or cloud services.

### CI and deployment separation

Continuous integration verifies that the repository is safe to build.

It does not:

- authenticate with Azure
- push images to Azure Container Registry
- create Azure resources
- deploy Container Apps
- run live inference
- run retraining
- promote models

Image publishing and deployment remain separate workflows with stricter
permissions.

## **10G.** Production container images

Phase 10G creates three independently deployable production images:

- FastAPI
- Streamlit
- batch pipeline

The images have separate dependency sets and runtime responsibilities.

The API and dashboard are HTTP services.

The pipeline image runs finite commands through Azure Container Apps Jobs.

### Container security and reproducibility

Each production image:

- uses Python 3.12
- installs from the locked uv dependency graph
- pins the uv tool version
- runs as a non-root user
- excludes local secrets
- records Git revision metadata
- uses a small Debian-based runtime
- separates dependency installation from source copying

The API and dashboard include health checks.

The pipeline image has no ingress.

### Image publication boundary

Phase 10G validates images locally only.

The images are not pushed to Azure Container Registry in this phase.

Phase 10H will:

1. authenticate with the existing registry
2. create immutable registry tags
3. build for `linux/amd64`
4. push API, dashboard, and pipeline images
5. verify registry digests

In [26]:
import json
from pathlib import Path


CONTAINER_REPORT_PATH = Path(
    PROJECT_ROOT/"reports/phase_10/container_image_validation_report.json"
)

container_image_report = json.loads(
    CONTAINER_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

container_image_report["status"]

'PRODUCTION_CONTAINER_IMAGES_VALIDATED'

In [27]:
assert container_image_report[
    "approved"
]

assert container_image_report[
    "status"
] == (
    "PRODUCTION_CONTAINER_IMAGES_VALIDATED"
)

assert container_image_report[
    "checks"
]["all_images_non_root"]

assert container_image_report[
    "checks"
]["api_healthcheck_configured"]

assert container_image_report[
    "checks"
]["dashboard_healthcheck_configured"]

assert container_image_report[
    "checks"
]["pipeline_has_no_healthcheck"]

print(
    "Phase 10G production container "
    "image validation passed."
)

Phase 10G production container image validation passed.


## **10H.** Azure Container Registry publishing

Phase 10H publishes the three validated production images to the existing
Azure Container Registry.

The images use isolated repositories:

- `pearls-aqi/api`
- `pearls-aqi/dashboard`
- `pearls-aqi/pipeline`

Each image is tagged using the complete Git commit SHA.

No floating `latest` tag is published.

### Publication safety

Registry publication and Azure deployment remain separate operations.

This phase:

1. authenticates with ACR
2. tags locally validated images
3. pushes immutable tags
4. verifies registry digests
5. records the publication report

No Container App or Container Apps Job is created or updated.

In [28]:
REGISTRY_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "registry_publication_report.json"
)

registry_publication_report = json.loads(
    REGISTRY_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

registry_publication_report["status"]

'REGISTRY_PUBLICATION_VALIDATED'

In [29]:
assert registry_publication_report[
    "approved"
]

assert registry_publication_report[
    "checks"
]["all_images_found"]

assert registry_publication_report[
    "checks"
]["all_digests_present"]

assert registry_publication_report[
    "checks"
]["all_digests_sha256"]

assert registry_publication_report[
    "checks"
]["tag_matches_git_commit"]

assert registry_publication_report[
    "checks"
]["tag_is_not_latest"]

assert not registry_publication_report[
    "floating_latest_tag_pushed"
]

assert not registry_publication_report[
    "deployment_performed"
]

print(
    "Phase 10H registry publication "
    "validation passed."
)

Phase 10H registry publication validation passed.


## **10I.** Staging infrastructure and deployment

Phase 10I deploys the immutable API and dashboard images to an isolated Azure
Container Apps staging environment.

The deployment uses:

- Azure Container Apps Consumption
- Central India
- scale-to-zero
- one maximum replica
- user-assigned managed identity
- private ACR image pulls
- private Blob Storage
- immutable image tags

Scheduled jobs are intentionally deferred to Phase 10J.

In [30]:
import json
from pathlib import Path


def find_project_root(
    start: Path | None = None,
) -> Path:
    current = (
        start or Path.cwd()
    ).resolve()

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            candidate
            / "pyproject.toml"
        ).exists():
            return candidate

    raise FileNotFoundError(
        "Project root could not be located."
    )


PROJECT_ROOT = find_project_root()

STAGING_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "staging_deployment_report.json"
)

staging_report = json.loads(
    STAGING_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

staging_report["status"]

'STAGING_DEPLOYMENT_VALIDATED'

## **10J.** Scheduled forecast and AQI publication

Phase 10J separates forecast generation from the serving applications.

The operational workflow is:

1. run live PM2.5 inference
2. preserve the exact Phase 5 run ID
3. run AQI and alert processing using that exact run
4. validate the complete Phase 6 package
5. publish an immutable artifact run
6. update the latest pointer only after publication succeeds
7. let FastAPI consume the latest successful package

The API and dashboard do not call OpenAQ, Open-Meteo, or the model during
normal user requests.

### Phase 10J-A — Local orchestration validation

This subphase validates the complete orchestration locally before Azure Blob
Storage and scheduling are enabled.

The local artifact repository is intentionally separate from `aqi/runs`.
This prevents the durable publication layer from copying files into their own
source directory.

In [3]:
import json

FORECAST_PUBLICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "forecast_publication_report.json"
)

forecast_publication_report = json.loads(
    FORECAST_PUBLICATION_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

forecast_publication_report["status"]

'FORECAST_PUBLICATION_COMPLETED'

### Phase 10J-B — Azure Blob publication

The locally validated publication workflow is now executed against Azure Blob
Storage.

Authentication remains passwordless:

- local validation uses the signed-in Azure CLI identity
- Azure-hosted jobs will use the user-assigned managed identity
- no storage account key or connection string is stored

The publication order remains:

1. upload the five validated Phase 6 artifacts
2. generate and upload the checksum manifest
3. update the latest pointer only after all preceding operations succeed

In [4]:
import json

FORECAST_PUBLICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "forecast_publication_report.json"
)

azure_publication_report = json.loads(
    FORECAST_PUBLICATION_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

azure_publication_report["status"]

'FORECAST_PUBLICATION_COMPLETED'

### Phase 10J-C — Blob-backed API artifact loading

The FastAPI service no longer requires the latest AQI forecast package to be
baked into its container image.

When the artifact backend is `azure_blob`, the API:

1. reads the durable `aqi/latest/pointer.json`
2. loads the immutable run manifest
3. confirms pointer and manifest lineage
4. downloads all manifest files
5. verifies every file size and SHA-256 checksum
6. atomically refreshes a writable local cache
7. applies the existing Phase 6 serving validations
8. serves the newly published forecast without rebuilding the API image

Local filesystem mode remains available for unit tests, CI fixtures, and local
development.

In [3]:
blob_cache_directory = (
    PROJECT_ROOT
    / ".cache"
    / "api"
    / "aqi"
    / "latest"
)

required_blob_cache_files = {
    "live_pm25_aqi_forecast.parquet",
    "alert_episodes.json",
    "aqi_forecast_summary.json",
    "aqi_metadata.json",
    "phase_6_validation_report.json",
}

actual_blob_cache_files = {
    path.name
    for path in blob_cache_directory.iterdir()
    if path.is_file()
}

assert (
    actual_blob_cache_files
    == required_blob_cache_files
)

print(
    "Phase 10J-C Blob-backed API "
    "materialization passed."
)

Phase 10J-C Blob-backed API materialization passed.


### Phase 10J-D — Blob-backed staging API

The Blob-backed artifact source was integrated into the deployed FastAPI
Container App.

The staging API now:

1. runs as a non-root container user
2. uses a writable local materialization cache
3. authenticates through a user-assigned managed identity
4. has read-only Blob data permissions
5. downloads the latest published AQI package
6. verifies its manifest, sizes, and SHA-256 checksums
7. serves the Blob run without rebuilding the API for each forecast

The storage account key and connection string are not used.

In [5]:
import json
from pathlib import Path


STAGING_FORECAST_PATH = Path(
    "/tmp/pearls-aqi-staging-blob.json"
)

STAGING_POINTER_PATH = Path(
    "/tmp/pearls-aqi-staging-pointer.json"
)

staging_forecast = json.loads(
    STAGING_FORECAST_PATH.read_text(
        encoding="utf-8"
    )
)

staging_pointer = json.loads(
    STAGING_POINTER_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(
    staging_forecast["hourly_forecast"]
) == 72

assert (
    staging_forecast["pipeline_run_id"]
    == staging_pointer["run_id"]
)

print(
    "Phase 10J-D passed. "
    "The staging API is serving:",
    staging_pointer["run_id"],
)

Phase 10J-D passed. The staging API is serving: 20260803T155020Z_aqi_9063b7ac


### Phase 10J-E — Scheduled forecast publication

An Azure Container Apps scheduled job was created for the complete forecast
publication workflow.

The job runs every six hours in UTC and performs:

1. live PM2.5 and weather retrieval
2. registry-backed model loading
3. 72-hour PM2.5 inference
4. AQI and alert generation
5. immutable artifact publication to Azure Blob Storage
6. atomic update of the latest-run pointer

The job runs as a finite batch workload and exits after publication. It uses
the staging user-assigned managed identity for ACR image pulls and Azure Blob
writes. OpenAQ and Hopsworks API keys are stored as job secrets rather than
committed configuration.

The deployed FastAPI service detects the new Blob pointer and serves the new
forecast without an API image rebuild or restart.

In [6]:
import json
from pathlib import Path


POINTER_PATH = Path(
    "/tmp/pearls-aqi-job-pointer.json"
)

READINESS_PATH = Path(
    "/tmp/pearls-aqi-job-readiness.json"
)

pointer = json.loads(
    POINTER_PATH.read_text(
        encoding="utf-8"
    )
)

readiness = json.loads(
    READINESS_PATH.read_text(
        encoding="utf-8"
    )
)

assert pointer["run_id"]
assert readiness["forecast_rows"] == 72

assert (
    pointer["run_id"]
    == readiness["pipeline_run_id"]
)

print(
    "Phase 10J-E passed for run:",
    pointer["run_id"],
)

Phase 10J-E passed for run: 20260804T084952Z_aqi_ebe06ce3


### Phase 10J-F — Controlled failure and recovery validation

A controlled Azure Container Apps Job failure was introduced using a separate
temporary manual job with an invalid OpenAQ credential.

The real scheduled forecast job was not modified.

The validation confirmed that:

1. the controlled execution failed as expected
2. no incomplete AQI package became the latest package
3. the durable Blob pointer remained on the last successful run
4. the staging API continued serving the last valid forecast
5. the real forecast job subsequently completed successfully
6. a new immutable run and manifest were published
7. the durable latest pointer advanced only after successful publication
8. the staging API automatically refreshed to the recovered run
9. no API image rebuild or deployment was required

This demonstrates fail-safe publication behavior: incomplete or failed
pipeline executions do not replace the currently served forecast.

In [7]:
import json
from pathlib import Path


RECOVERY_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "forecast_job_recovery_report.json"
)

recovery_report = json.loads(
    RECOVERY_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    recovery_report["status"]
    == "FORECAST_JOB_FAILURE_RECOVERY_VALIDATED"
)

assert (
    recovery_report[
        "failure_execution"
    ]["pointer_unchanged"]
)

assert (
    recovery_report[
        "recovery_execution"
    ]["pointer_advanced"]
)

assert (
    recovery_report[
        "api_validation"
    ][
        "served_last_valid_run_after_failure"
    ]
)

assert (
    recovery_report[
        "api_validation"
    ][
        "automatically_served_recovered_run"
    ]
)

assert (
    recovery_report[
        "api_validation"
    ]["forecast_rows"]
    == 72
)

print(
    "Phase 10J-F failure and recovery "
    "validation passed."
)

Phase 10J-F failure and recovery validation passed.


### Phase 10J-G — Final forecast automation report

Phase 10J consolidated the complete scheduled forecast-publication path.

The deployed operational architecture now consists of:

1. an Azure Container Apps scheduled job
2. live OpenAQ and Open-Meteo data retrieval
3. Hopsworks production-model loading
4. 72-hour PM2.5 inference
5. AQI and alert generation
6. immutable Azure Blob artifact publication
7. manifest and SHA-256 validation
8. an atomic latest-run pointer
9. a Blob-backed FastAPI service
10. automatic API refresh without redeployment

A controlled failure test confirmed that unsuccessful executions cannot replace
the last valid forecast. A subsequent recovery execution published a new run,
advanced the pointer, and was automatically served by the API.

The operational commands, permissions, artifact layout, failure behavior, and
recovery procedure are documented in:

`docs/forecast_automation_runbook.md`

In [8]:
import json
from pathlib import Path


FINAL_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "forecast_automation_final_report.json"
)

final_report = json.loads(
    FINAL_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

assert final_report["phase"] == "10J"

assert (
    final_report["status"]
    == "PHASE_10J_FORECAST_AUTOMATION_APPROVED"
)

assert final_report["approved"] is True

assert all(
    details["status"] == "COMPLETED"
    for details in final_report[
        "subphases"
    ].values()
)

assert all(
    final_report[
        "recovery_checks"
    ].values()
)

assert (
    final_report[
        "latest_validated_recovery"
    ]["forecast_rows"]
    == 72
)

print(
    "Phase 10J completed and approved."
)

Phase 10J completed and approved.


## **10K** Production hourly feature pipeline

The original incremental feature synchronization command operated on the
existing local Phase 1 and Phase 2 Parquet artifacts. This was appropriate for
validating incremental Hopsworks behavior, but it was not sufficient for a
scheduled production job because those local files would not automatically
contain newly published source observations.

### Phase 10K-A Introduces a fresh-data orchestration layer:

```text
OpenAQ recent hourly PM2.5
        +
Open-Meteo recent observed weather
        ↓
Safe reference-time selection
        ↓
Recent aligned source window
        ↓
Reusable reference-time feature construction
        ↓
Incremental Hopsworks synchronization

In [1]:
import os
import subprocess
from pathlib import Path


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


environment = os.environ.copy()

environment.update(
    {
        "FEATURE_STORE_BACKEND": "hopsworks",
        "MODEL_REGISTRY_BACKEND": "hopsworks",
        "MLOPS_DRY_RUN": "true",
    }
)

result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-m",
        "app.pipelines.hourly_features",
    ],
    cwd=PROJECT_ROOT,
    env=environment,
    check=False,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0

Observed PM2.5 rows: 71
Observed PM2.5 range: 2026-08-02 10:00:00+00:00 to 2026-08-05 08:00:00+00:00
Missing PM2.5 values: 0
Missing PM2.5 hours: 0
[]
Engineered rows: 71
Reference range: 2026-08-02 10:00:00+00:00 to 2026-08-05 08:00:00+00:00
Columns with missing values:
pm25_change_24h    24
pm25_lag_24h       24
pm25_mean_24h      23
pm25_lag_12h       12
pm25_mean_12h      11
pm25_lag_6h         6
pm25_change_6h      6
pm25_mean_6h        5
pm25_lag_3h         3
pm25_mean_3h        2
pm25_change_1h      1
pm25_lag_1h         1

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41128
Reading data from Hopsworks, using Hopsworks Feature Query Service.   
Reading data from Hopsworks, using Hopsworks Feature Query Service..   
Reading data from Hopsworks, using Hopsworks Feature Query Service...   
Reading data from Hopsworks, using Hopsworks Feature Query Service   
Reading data from Hopsworks, using Hopsworks Feature Query Service.   
Reading data from Hop

In [ ]:
import json

REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "hourly_feature_pipeline_report.json"
)

hourly_feature_report = json.loads(
    REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    hourly_feature_report["phase"]
    == "10K"
)

assert (
    hourly_feature_report["subphase"]
    == "10K-A"
)

assert hourly_feature_report[
    "status"
] in {
    (
        "HOURLY_FEATURE_PIPELINE_"
        "DRY_RUN_COMPLETED"
    ),
    (
        "HOURLY_FEATURE_PIPELINE_"
        "COMPLETED"
    ),
    (
        "HOURLY_FEATURE_PIPELINE_"
        "NO_CHANGES"
    ),
}

assert hourly_feature_report[
    "reference_selection"
][
    "selected_reference_time"
]

assert hourly_feature_report[
    "source_data"
][
    "complete_engineered_rows"
] > 0

assert hourly_feature_report[
    "validation"
][
    "future_weather_excluded"
] is True

assert hourly_feature_report[
    "validation"
][
    "contract_ordering_applied"
] is True

print(
    "Phase 10K-A validated:",
    hourly_feature_report["status"],
)

Phase 10K-A validated: HOURLY_FEATURE_PIPELINE_DRY_RUN_COMPLETED


### Phase 10K-B — Scheduled hourly feature synchronization

The validated fresh-data feature pipeline was deployed as an Azure Container
Apps scheduled job.

The job executes:

```text
python -m app.pipelines.hourly_features

Its UTC schedule is:

15 * * * *

The job starts at minute 15 of every hour. This small delay avoids assuming
that upstream hourly source data is finalized exactly at minute zero.

Each execution:

fetches recent OpenAQ PM2.5 observations
fetches recent Open-Meteo weather
selects the latest safe reference hour
excludes future forecast weather from historical observations
constructs complete reusable reference-time features
compares the overlap window with Hopsworks
writes only inserted or changed rows
exits successfully when no changes are required

The job uses an immutable pipeline image and stores OpenAQ and Hopsworks API
keys as Container Apps secret references.

In [6]:
import json
from pathlib import Path


HOURLY_JOB_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "phase_10"
    / "hourly_feature_job_report.json"
)

hourly_job_report = json.loads(
    HOURLY_JOB_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

assert hourly_job_report[
    "status"
] == "HOURLY_FEATURE_JOB_VALIDATED"

assert hourly_job_report["approved"]

assert hourly_job_report[
    "schedule_utc"
] == "15 * * * *"

assert all(
    hourly_job_report[
        "checks"
    ].values()
)

assert not hourly_job_report[
    "daily_retraining_job_created"
]

assert not hourly_job_report[
    "automatic_model_promotion_enabled"
]

print(
    "Phase 10K-B hourly feature job "
    "validation passed."
)

Phase 10K-B hourly feature job validation passed.


### Phase 10K-C1 — Fresh production training dataset

The static Phase 2 Parquet datasets were suitable for reproducible initial
model training, but they could not include newly synchronized Hopsworks data.

The production refresh pipeline now reads:

- PM2.5 observations from Hopsworks
- historical weather observations from Hopsworks
- reusable reference-time features from Hopsworks

It recreates the approved direct multi-horizon training structure:

```text
reference features
        ↓
horizons 1–72
        ↓
target timestamps
        ↓
target PM2.5 and target weather
        ↓
target calendar and wind features
        ↓
completeness and leakage validation
        ↓
chronological 70/15/15 splits
        ↓
72-hour split purge

### Phase 10K-C2 — Runtime retraining eligibility

The production retraining command was updated to consume an immutable runtime
training package rather than the original static Phase 2 Parquet files.

Retraining eligibility now compares the newest fully labeled reference
timestamp with the latest dataset boundary already consumed during production
model selection and approval.

The consumed-data boundary is resolved in this order:

1. explicit production-data end
2. test reference end
3. validation reference end
4. training reference end

This prevents validation and test data that already influenced model selection
from being incorrectly treated as new retraining data.

For the current production model:

```text
production consumed-data end: 2026-07-23 22:00 UTC
latest fully labeled runtime reference: 2026-07-20 23:00 UTC
new labeled reference hours: 0

### Phase 10K-C3 — Production-safe daily retraining orchestrator

The production retraining workflow now has one non-interactive entrypoint:

```text
python -m app.pipelines.daily_retraining

### Phase 10K-D — Scheduled Azure retraining job

The production-safe retraining workflow was packaged into the shared pipeline
image and deployed as an Azure Container Apps scheduled job.

Configuration:

```text
job name: job-pearls-aqi-retraining
schedule: 30 3 * * *
timezone: UTC
parallelism: 1
completion count: 1
timeout: 3600 seconds
retry limit: 1
CPU: 1 core
memory: 2 GiB

### Phase 10K-E — Daily retraining operational validation

The scheduled retraining job is validated through a repeatable Python
validator rather than manual portal inspection.

The validator checks:

- successful Azure provisioning
- scheduled trigger configuration
- daily UTC cron expression
- one replica and one completion
- timeout and retry settings
- immutable image reference
- exact daily retraining wrapper command
- CPU and memory configuration
- most recent successful execution

Normal operational outcomes:

```text
DAILY_RETRAINING_SKIPPED

## **10L-A.** Read-only production health snapshot

A production health command now inspects the operational state of the deployed
system without modifying any resources or data.

The health snapshot covers:

- latest hourly feature-job execution
- latest six-hour forecast-publication execution
- latest daily retraining execution
- latest PM2.5 observation in Hopsworks
- latest historical weather observation in Hopsworks
- latest engineered reference feature in Hopsworks
- latest approved AQI artifact pointer in Azure Blob Storage

Each component is assigned one of four states:

```text
HEALTHY
WARNING
CRITICAL
UNKNOWN

### Phase 10L-B — Monitoring-rule validation

The monitoring severity rules were validated independently from live production
resources.

Tests cover:

- exact healthy, warning, and critical age boundaries
- missing timestamps
- future timestamps caused by minor clock skew
- overall worst-status selection
- public report-status mapping
- invalid threshold configuration

This allows stale-data and failure behavior to be tested without intentionally
delaying production pipelines, modifying Hopsworks data, or failing Azure jobs.

### Phase 10L-D1 — Durable health history and incident deduplication

Every hourly monitoring execution now publishes an immutable production-health
snapshot through the established artifact repository.

Storage layout:

```text
production-health/runs/<health-run-id>/
production-health/latest/pointer.json
production-health/incidents/active.json
production-health/incidents/history/<event-id>.json

### Phase 10L-D2 — Production health persistence on Azure

The scheduled production-monitoring job now runs the durable health
orchestrator rather than the temporary read-only command.

Each execution:

1. inspects Azure job executions;
2. reads production Hopsworks feature timestamps;
3. validates the latest AQI artifact;
4. publishes an immutable production-health snapshot;
5. validates the new manifest and latest pointer;
6. maintains deduplicated incident state.

Azure Blob layout:

```text
production-health/runs/<health-run-id>/
production-health/latest/pointer.json
production-health/incidents/active.json
production-health/incidents/history/<event-id>.json